# Ai 智能体的记忆机制

## 记忆类型
* 短期记忆：
    * 用于存储当前任务的相关信息
    * 例如：当前用户的问题、当前的操作步骤等
* 长期记忆：
> 使得AI agent突破单次对话的局限，构建持久、个性化的知识体系
    * 用于存储长期的知识和经验
    * 例如：用户的历史问题、用户的操作记录等


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

True

## 1.在 LangGraph 中通过对话历史记录管理短期记忆，通过中间结果管理短期记忆

In [3]:
from typing import TypedDict, List, Dict, Any
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
# 定义 AgentState 结构来存储对话历史记录：
class AgentState(TypedDict):
    messages: List[BaseMessage]  # 消息列表，作为对话历史记录
    intermediate_results: Dict[str, Any]  # 短期记忆中的其他数据

In [5]:
def add_user_message(state: AgentState, user_message: str) -> Dict[str, List[BaseMessage]]:
    """向对话历史记录添加用户消息"""
    new_message = HumanMessage(content=user_message)
    return {"messages": state["messages"] + [new_message]}

def add_ai_message(state: AgentState, ai_response: str) -> Dict[str, List[BaseMessage]]:
    """向对话历史记录添加 AI 响应"""
    new_message = AIMessage(content=ai_response)
    return {"messages": state["messages"] + [new_message]}

# 测试短期记忆管理
initial_state = AgentState(messages=[], intermediate_results={})
state_with_user_msg = add_user_message(initial_state, "你好，我是用户")
print("添加用户消息后的状态：", len(state_with_user_msg["messages"]), "条消息")

final_state = add_ai_message(state_with_user_msg, "你好！我是AI助手，很高兴为您服务")
print("添加AI消息后的状态：", len(final_state["messages"]), "条消息")

添加用户消息后的状态： 1 条消息
添加AI消息后的状态： 2 条消息


## 消息的截断
随着对话的延长，需要管理内存和成本。这里演示两种截断策略：
1. 保留最新消息，可以按消息数量和token
    * 固定长度截断
    * 基于令牌长度截断

2. 历史记录的摘要
    降低内存成本，保留对话的核心信息


### demo1：消息的截断 

In [9]:
from langchain_core.messages import trim_messages
from langchain_openai import ChatOpenAI

def truncate_history(state: AgentState, max_messages: int) -> Dict[str, List[BaseMessage]]:
    """截断对话历史记录, 保留最新的 max_messages 条消息"""
    messages = state["messages"]
    if len(messages) <= max_messages:
        return state
    truncated_messages = messages[-max_messages:]
    return {"messages": truncated_messages}

def trim_message_history_by_token(state: AgentState, max_tokens: int) -> Dict[str, List[BaseMessage]]:
    """根据令牌长度截断对话历史记录
    使用langchain的trim_messages函数根据词元计数修剪消息历史记录
    """
    messages = state["messages"]
    trimmed_messages = trim_messages(
        messages,
        strategy="last",
        token_counter=ChatOpenAI(model=os.getenv("MODEL_NAME")),
        max_tokens=max_tokens,
        start_on="human",
        end_on=("human", "tool"),
        include_end=True,   # 保留系统消息
    )
    return {"messages": trimmed_messages}

# 创建一个包含多条消息的测试状态
test_state = AgentState(
    messages=[
        HumanMessage(content="第一条用户消息"),
        AIMessage(content="第一条AI回复"),
        HumanMessage(content="第二条用户消息"),
        AIMessage(content="第二条AI回复"),
        HumanMessage(content="第三条用户消息"),
        AIMessage(content="第三条AI回复")
    ],
    intermediate_results={}
)

# 测试消息数量截断
truncated_state = truncate_history(test_state, 2)
print(f"原始消息数量: {len(test_state['messages'])}")
print(f"截断后消息数量: {len(truncated_state['messages'])}")

# 显示截断后的消息内容
for i, msg in enumerate(truncated_state['messages']):
    print(f"消息 {i+1}: {type(msg).__name__} - {msg.content}")
    

原始消息数量: 6
截断后消息数量: 2
消息 1: HumanMessage - 第三条用户消息
消息 2: AIMessage - 第三条AI回复


### demo2：摘要 
核心就是将消息交给LLM，让LLM来总结消息的内容

# 记忆的存储

BaseStore最基础的抽象，可以基于此实现不同的记忆存储方式，例如：
* 内存存储
* 文件存储
* 数据库存储
* 云存储等

核心流程：
1. 定义命名空间来组织记忆
2. 为每个记忆条目生成唯一键
3. 创建字典保存记忆内容
4. 使用 put 方法将记忆存储在命名空间中
5. 使用 get / search 方法从命名空间中检索记忆
    * get：基于精确键值的直接检索
    * search：支持语义、内容过滤的检索

In [12]:
import uuid
from langgraph.store.memory import InMemoryStore

in_memory_store = InMemoryStore()

# 定义用户特定数据的命名空间
user_id = "example_user"
namespace_for_user_data = (user_id, "user_info")

# 为记忆条目生成唯一键
memory_key = str(uuid.uuid4())

# 创建一个字典来保存用户姓名作为记忆值
memory_value = {"user_name": "xing"}

# 使用 put 将记忆存储在 InMemoryStore 中
in_memory_store.put(namespace_for_user_data, memory_key, memory_value)

print(f"记忆已保存，键为：{memory_key}，命名空间为：{namespace_for_user_data}")

记忆已保存，键为：88e42eb5-dfcf-45b8-ae13-9422ec515d8a，命名空间为：('example_user', 'user_info')


In [13]:
# 检索 user_info 命名空间中的所有记忆（没有查询或过滤器）
all_user_memories = in_memory_store.search(namespace_for_user_data)
print("命名空间中的所有用户记忆：")
for record in all_user_memories:
    print(record.dict()) # 打印 MemoryRecord 字典表示

# 使用 'get' 通过键检索记忆
retrieved_memory_record = in_memory_store.get(namespace_for_user_data, memory_key)
print(f"\n使用键 '{memory_key}' 检索到的记忆：")
print(retrieved_memory_record.dict()) # 打印 MemoryRecord 字典表示

命名空间中的所有用户记忆：
{'namespace': ['example_user', 'user_info'], 'key': '88e42eb5-dfcf-45b8-ae13-9422ec515d8a', 'value': {'user_name': 'xing'}, 'created_at': '2025-08-31T13:40:43.827695+00:00', 'updated_at': '2025-08-31T13:40:43.827697+00:00', 'score': None}

使用键 '88e42eb5-dfcf-45b8-ae13-9422ec515d8a' 检索到的记忆：
{'namespace': ['example_user', 'user_info'], 'key': '88e42eb5-dfcf-45b8-ae13-9422ec515d8a', 'value': {'user_name': 'xing'}, 'created_at': '2025-08-31T13:40:43.827695+00:00', 'updated_at': '2025-08-31T13:40:43.827697+00:00'}


## 基于向量检索的记忆

In [15]:
from langchain_openai import OpenAIEmbeddings
from langgraph.store.memory import InMemoryStore

# 初始化 BGE-M3 向量化模型
embeddings = OpenAIEmbeddings(model="BAAI/bge-m3") # 或其他向量化模型

# 使用语义搜索索引配置 InMemoryStore
store_with_semantic_search = InMemoryStore(
    index={
        "embed": embeddings.embed_documents, 
        "dims": 1024, # BGE-M3 的向量维度
        "fields": ["memory_content"] # 仅向量化 "memory_content" 字段（可选）
    }
)

# 保存将为语义搜索索引的记忆（默认行为）
store_with_semantic_search.put(
    ("user_789", "food_memories"),
    "memory_1",
    {"memory_content": "我真的很喜欢辛辣的印度咖喱。"},
)

# 保存另一个记忆，显式禁用此条目的索引
store_with_semantic_search.put(
    ("user_789", "system_metadata"),
    "memory_2",
    {"memory_content": "用户入职已完成。", "status": "completed"},
    index=False, # 禁用此记忆的索引
)

# 保存一个记忆，覆盖默认索引字段并仅索引 "context"
store_with_semantic_search.put(
    ("user_789", "restaurant_reviews"),
    "memory_3",
    {"memory_content": "服务很慢，但食物很好。", "context": "对 'The Italian Place' 餐厅的评论"},
    index=["context"] # 仅索引 "context" 字段
)

# 语义搜索食物偏好
search_query = "该用户喜欢哪种食物？"
semantic_memory_results = store_with_semantic_search.search(
    ("user_789", "food_memories"), query=search_query, limit=2
)

print("查询的语义搜索结果：", search_query)
for record in semantic_memory_results:
    print(f"记忆键：{record.key}，相似度评分：{record.score}")
    print(f"记忆内容：{record.value}")
    print("=" * 30)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-52338***********************daf8. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}